# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/02017711723iot-dotcom/Flyrank_Internship_1/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
# Install the packages needed for the warehouse
%pip install -q duckdb huggingface_hub pandas scikit-learn

import os
import duckdb
import pandas as pd
import numpy as np

print("Packages loaded successfully.")

Packages loaded successfully.


In [2]:
# Get the Hugging Face token from Colab Secrets
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

if not HF_TOKEN:
    raise ValueError(
        "HF_TOKEN was not found. Add your Hugging Face READ token "
        "to Colab Secrets with the name HF_TOKEN."
    )

# Create DuckDB connection
con = duckdb.connect()

# Authenticate DuckDB with Hugging Face
con.execute(
    f"CREATE OR REPLACE SECRET hf_secret "
    f"(TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

print("Hugging Face connection is ready.")

Hugging Face connection is ready.


In [3]:
# Hugging Face warehouse location
HF_WAREHOUSE = "hf://datasets/FlyRank/internship-warehouse"

# Main daily performance table
DAILY_TABLE = (
    f"{HF_WAREHOUSE}/fact_content_daily_performance/**/*.parquet"
)

print("Warehouse configured.")
print("Table:", DAILY_TABLE)

Warehouse configured.
Table: hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet


In [4]:
# Look at the columns and data types
schema_df = con.execute(
    f"""
    DESCRIBE
    SELECT *
    FROM read_parquet('{DAILY_TABLE}', hive_partitioning=true)
    LIMIT 1
    """
).df()

display(schema_df)

,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*
One row represents one webpage for a particular month.I will use the warehouse table containing page-level search and content performance data for my Content Refresh Prioritization lane. I will use the March 2026 data from this table for my analysis.

Time Window:March 2026.

I want to rank webpages according to their likelihood of showing a declining search trend, so that content teams can prioritize pages for review.

I will deliberately exclude any feature that is derived from the future outcome or from a decision that was made after the prediction moment.



In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Fields: feature / label / context / excluded
## My Data Contract

### 1. What one row means

One row represents one webpage for a particular month.

### 2. Tables I will use

I will use the warehouse table containing the page-level search/content observations required for my Content Refresh Prioritization lane.

### 3. Time window

I will use March 2026 as my development month. I will avoid using the final June 2026 month for developing the label or feature logic.

### 4. Prediction / ranking target

I want to rank webpages according to their likelihood of showing a declining search trend, so that content teams can prioritize pages for review.

### 5. Deliberate exclusion

I will exclude features that contain future information or information created after the prediction decision, because they could leak the outcome into the model.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [7]:
# QUERY 1 — Verify the grain
# One row should represent one client + one content item + one report date.

grain_check = con.execute(
    f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(
            DISTINCT
            CAST(report_date AS VARCHAR)
            || '|' ||
            client_hash_id
            || '|' ||
            content_hash_id
        ) AS distinct_grain_keys
    FROM read_parquet(
        '{DAILY_TABLE}',
        hive_partitioning=true
    )
    WHERE month = '2026-03'
    """
).df()

display(grain_check)

total_rows = grain_check.loc[0, "total_rows"]
distinct_keys = grain_check.loc[0, "distinct_grain_keys"]

if total_rows == distinct_keys:
    print("PASS: One row is uniquely identified by report_date + client_hash_id + content_hash_id.")
else:
    print("CHECK: Duplicate grain keys were found.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,distinct_grain_keys
0,9841378,9841378


PASS: One row is uniquely identified by report_date + client_hash_id + content_hash_id.


In [8]:
# QUERY 2
# Count the rows in March 2026 and show the first and last report dates.

row_date_check = con.execute(
    f"""
    SELECT
        COUNT(*) AS row_count,
        MIN(report_date) AS first_date,
        MAX(report_date) AS last_date
    FROM read_parquet(
        '{DAILY_TABLE}',
        hive_partitioning=true
    )
    WHERE month = '2026-03'
    """
).df()

display(row_date_check)

print("March 2026 row count:", row_date_check.loc[0, "row_count"])
print("First report date:", row_date_check.loc[0, "first_date"])
print("Last report date:", row_date_check.loc[0, "last_date"])

,row_count,first_date,last_date
0,9841378,2026-03-01,2026-03-31


March 2026 row count: 9841378
First report date: 2026-03-01 00:00:00
Last report date: 2026-03-31 00:00:00


In [9]:
# QUERY 3
# Count March 2026 rows where Search Console data is available.
# The assignment specifically requires IS TRUE.

availability_check = con.execute(
    f"""
    SELECT
        COUNT(*) AS available_rows
    FROM read_parquet(
        '{DAILY_TABLE}',
        hive_partitioning=true
    )
    WHERE month = '2026-03'
      AND client_has_gsc IS TRUE
    """
).df()

display(availability_check)

print(
    "Rows with Search Console availability:",
    availability_check.loc[0, "available_rows"]
)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,available_rows
0,9841378


Rows with Search Console availability: 9841378


## 4. Data limits
This data can help identify webpages that may need review, but it cannot prove that updating a page will cause its search performance to improve. The history may be unbalanced across clients, some early records may contain only Google Search Console data, and rolling time windows may overlap. Therefore, the data can support prioritization and prediction, but it cannot by itself establish causation.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [✅] Every section above is filled — markdown thinking AND the code that backs it
- [✅] The notebook runs top to bottom with no errors (Runtime → Run all)
- [✅] No client names, URLs, or private queries anywhere
- [✅] My claims use careful words: observed, measured, directional, decision-support
- [✅] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.